In [1]:
import glob  # 用于查找符合特定规则的文件路径名
from tqdm import tqdm  # 用于显示循环进度条
from pathlib import Path  # 提供面向对象的文件系统路径操作
from shutil import copyfile  # 用于文件的复制操作
import os  # 提供与操作系统交互的功能，如文件和目录操作
import subprocess  # 用于执行外部命令和子进程管理

In [2]:
base_url = r"C:\Users\cdh96\Desktop\demo-3d-reconstruction\images"
output_url = r"C:\Users\cdh96\Desktop\demo-3d-reconstruction\output"
dataset_url = r"C:\Users\cdh96\Desktop\demo-3d-reconstruction\dataset"
realityCapture_url = r"C:\Program Files\Capturing Reality\RealityCapture\RealityCapture.exe"
realityCapture_params_url = r"D:\lab\paper\2-3d_reconstruction\params.xml"
img_num = 150

## 创建项目

In [8]:
current_url = ''
temp_num = 0
for image_url in tqdm(glob.glob(os.path.join(base_url, '*.JPG'))):
    image_name = Path(image_url).stem.split('_')[1]

    if temp_num % img_num == 0:
        project_url = os.path.join(output_url, image_name)
        
        if os.path.exists(project_url):
            print(f"文件夹 {project_url} 已存在")
            current_url = project_url  
        else:
            os.mkdir(project_url)
            os.mkdir(os.path.join(project_url, 'labels'))
            os.mkdir(os.path.join(project_url, 'project'))
            os.mkdir(os.path.join(project_url, 'original'))
            os.mkdir(os.path.join(project_url, 'fore'))
            current_url = project_url

    if os.path.exists(os.path.join(current_url, 'original')):
        destination_file = os.path.join(current_url, 'original', Path(image_url).name)
        copyfile(image_url, destination_file)
    temp_num += 1

100%|██████████| 300/300 [00:00<00:00, 418.88it/s]


## 前景提取

In [3]:

for image_url in tqdm(glob.glob(os.path.join(output_url, '*', 'original'))):
    print(f"正在处理图像: {image_url}")
    result = subprocess.run([
        "rembg", "p", "-m", "birefnet-general", "-x",
        r'{"model_path": "C:\Users\cdh96\.u2net\birefnet-general.onnx"}',
        image_url,
        image_url.replace('original', 'fore')
    ], capture_output=True, text=True)

  0%|          | 0/2 [00:00<?, ?it/s]

正在处理图像: C:\Users\cdh96\Desktop\demo-3d-reconstruction\output\0429\original


 50%|█████     | 1/2 [29:26<29:26, 1766.42s/it]

正在处理图像: C:\Users\cdh96\Desktop\demo-3d-reconstruction\output\0729\original


100%|██████████| 2/2 [58:38<00:00, 1759.38s/it]


## 三维重建

In [4]:
for project_url in tqdm(glob.glob(os.path.join(output_url,'*'))):

    project_name = Path(project_url).name
    subprocess.run([realityCapture_url,
                    "-addFolder",os.path.join(project_url,'fore'),
                    "-align",
                    "-mergeComponents",
                    "-setReconstructionRegionAuto",
                    "-calculateHighModel",
                    "-calculateVertexColors",
                    "-calculateTexture",
                    "-exportModel","Model 1",os.path.join(project_url,'labels',f"{project_name}.ply"),realityCapture_params_url,
                    "-save", os.path.join(project_url,
                    'project',f"{project_name}.rcproj"),
                    "-quit"
                    ])



100%|██████████| 2/2 [16:02<00:00, 481.20s/it]


## 三维预处理

In [5]:
import open3d as o3d

for mesh_url in tqdm(glob.glob(os.path.join(output_url, '*', 'labels', '*.obj'))):

    mesh = o3d.io.read_triangle_mesh(mesh_url)

    # 先进行均匀采样
    pcd = mesh.sample_points_uniformly(number_of_points=10000)
    # 使用半径滤波去除离群点
    pcd, ind = pcd.remove_radius_outlier(nb_points=3, radius=0.05)

    mesh_path_obj = Path(mesh_url)
    ply_filename = mesh_path_obj.with_suffix('.ply').name
    dataset_ply_path = Path(dataset_url) / ply_filename
    Path(dataset_url).mkdir(parents=True, exist_ok=True)
    o3d.io.write_point_cloud(str(dataset_ply_path), pcd)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


100%|██████████| 2/2 [00:11<00:00,  5.90s/it]


## 全局配置

In [ ]:
from pathlib import Path
import os

BASE        = Path(r"C:\Users\26457\Desktop\demo-3d-reconstruction")
VIDEO_DIR   = BASE / "video"       # 输入视频目录
IMAGE_DIR   = BASE / "images"      # 提取的帧 / 原始图像
OUTPUT_DIR  = BASE / "output"      # 重建输出
DATASET_DIR = BASE / "dataset"     # 清洗后的PLY
RESULT_DIR  = BASE / "results"     # 最终指标

ARUCO_MARKER_SIZE_M = 0.10         # ArUco标记实际边长（米）
ARUCO_DICT          = "DICT_4X4_50"
IMG_PER_PROJECT     = 150

for d in [VIDEO_DIR, IMAGE_DIR, OUTPUT_DIR, DATASET_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("目录初始化完成")


## 视频帧提取

In [ ]:
import cv2
from tqdm import tqdm

def extract_frames(video_path: Path, output_dir: Path, interval: int = 10) -> int:
    """
    每隔 interval 帧提取一帧保存为 JPG。
    interval=10 @ 30fps => 每 0.33s 一帧。
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"无法打开视频: {video_path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = saved = 0
    with tqdm(total=total, desc=video_path.name) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % interval == 0:
                cv2.imwrite(str(output_dir / f"{video_path.stem}_{saved:05d}.JPG"), frame)
                saved += 1
            frame_idx += 1
            pbar.update(1)
    cap.release()
    return saved

video_files = (
    list(VIDEO_DIR.glob("*.mp4")) +
    list(VIDEO_DIR.glob("*.avi")) +
    list(VIDEO_DIR.glob("*.mov"))
)
if video_files:
    for vf in video_files:
        n = extract_frames(vf, IMAGE_DIR / vf.stem, interval=10)
        print(f"{vf.name} -> 提取 {n} 帧")
else:
    print("VIDEO_DIR 中无视频，跳过帧提取（使用已有图像）")


## 尺度校准（ArUco 俯视图检测）

In [ ]:
import numpy as np
import open3d as o3d
import cv2
import cv2.aruco as aruco

def calibrate_scale_aruco(
    ply_path: str,
    marker_size_m: float,
    dict_name: str = "DICT_4X4_50",
    img_px: int = 2000,
) -> float:
    """
    将点云俯视投影为彩色图像，检测ArUco，
    由标记四边长推算 PLY单位 -> 米 的缩放因子。
    """
    pcd  = o3d.io.read_point_cloud(ply_path)
    pts  = np.asarray(pcd.points)
    cols = (np.asarray(pcd.colors) * 255).astype(np.uint8)

    x0, y0 = pts[:, 0].min(), pts[:, 1].min()
    span = max(pts[:, 0].max() - x0, pts[:, 1].max() - y0)
    if span == 0:
        raise ValueError("点云为空或退化")
    ppu = img_px / span  # pixels per PLY-unit

    img = np.full((img_px, img_px, 3), 255, dtype=np.uint8)
    xi = np.clip(((pts[:, 0] - x0) * ppu).astype(int), 0, img_px - 1)
    yi = np.clip(((pts[:, 1] - y0) * ppu).astype(int), 0, img_px - 1)
    img[yi, xi] = cols[:, :3]

    gray = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_RGB2GRAY)
    adict    = aruco.getPredefinedDictionary(getattr(aruco, dict_name))
    detector = aruco.ArucoDetector(adict, aruco.DetectorParameters())
    corners, ids, _ = detector.detectMarkers(gray)

    if ids is None or len(ids) == 0:
        raise RuntimeError(
            "俯视图中未检测到ArUco。\n"
            "请确认: 1)点云含颜色 2)标记朝上可见 3)dict_name与实际一致"
        )

    c = corners[0][0]  # (4, 2) 像素坐标
    side_px = [np.linalg.norm(c[(i+1)%4] - c[i]) for i in range(4)]
    marker_size_ply = np.mean(side_px) / ppu
    scale = marker_size_m / marker_size_ply

    print(f"  检测到标记 ID={ids.flatten()}")
    print(f"  PLY边长={marker_size_ply:.5f}  实际={marker_size_m}m  缩放因子={scale:.4f}")

    # 保存可视化
    vis = img.copy()
    aruco.drawDetectedMarkers(vis, corners, ids)
    cv2.imwrite(ply_path.replace(".ply", "_aruco_topview.png"),
                cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return scale


## 地面识别 & 植物分割

In [ ]:
def detect_ground(
    pcd: o3d.geometry.PointCloud,
    distance_threshold: float = 0.02,
    ransac_n: int = 3,
    num_iterations: int = 1000,
):
    """
    RANSAC平面拟合，返回 (plane_model, ground_pcd, above_pcd)。
    distance_threshold 单位与PLY相同（校准前原始单位）。
    """
    model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations,
    )
    a, b, c, d = model
    print(f"  地面: 法向量=({a:.3f},{b:.3f},{c:.3f}) d={d:.3f} 内点={len(inliers)}")
    return model, pcd.select_by_index(inliers), pcd.select_by_index(inliers, invert=True)


def segment_plant(
    above: o3d.geometry.PointCloud,
    eps: float = 0.05,
    min_points: int = 20,
) -> o3d.geometry.PointCloud:
    """
    DBSCAN聚类，返回最大连通体（即植物主体）。
    eps 单位与PLY相同。
    """
    labels = np.array(
        above.cluster_dbscan(eps=eps, min_points=min_points, print_progress=False)
    )
    if labels.max() < 0:
        raise RuntimeError("DBSCAN未找到有效簇，请调整 eps 或 min_points")
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)
    best = unique[np.argmax(counts)]
    idx  = np.where(labels == best)[0]
    print(f"  {labels.max()+1} 个簇，主体含 {len(idx)} 个点")
    return above.select_by_index(idx)


## 表型指标计算（株高 / 冠幅 / 体积 / 叶面积）

In [ ]:
from scipy.spatial import ConvexHull

def compute_traits(
    plant_pcd: o3d.geometry.PointCloud,
    plane_model: list,
    scale_factor: float,
) -> dict:
    """
    计算植物表型指标，所有结果单位为米。
    """
    pts = np.asarray(plant_pcd.points) * scale_factor
    a, b, c, d = plane_model
    # 平面方程系数同步缩放（d 是截距，不乘 scale）
    a, b, c = a / scale_factor, b / scale_factor, c / scale_factor
    normal_len = np.sqrt(a**2 + b**2 + c**2)

    # 株高：最高点到地面的距离
    dist = (pts @ np.array([a, b, c]) + d) / normal_len
    height_m = float(dist.max() - dist.min())

    # 冠幅：投影到水平面后凸包最大直径
    n_hat = np.array([a, b, c]) / normal_len
    proj  = pts - np.outer(dist, n_hat)
    xy    = proj[:, :2]
    try:
        h2d     = ConvexHull(xy)
        hpts    = xy[h2d.vertices]
        canopy_m = float(np.linalg.norm(hpts[:, None] - hpts[None, :], axis=-1).max())
    except Exception:
        canopy_m = float(np.ptp(xy, axis=0).max())

    # 体积：3D凸包
    try:
        volume_m3 = float(ConvexHull(pts).volume)
    except Exception:
        volume_m3 = -1.0

    # 叶面积：Poisson重建后的网格表面积
    pcd_m = o3d.geometry.PointCloud()
    pcd_m.points = o3d.utility.Vector3dVector(pts)
    pcd_m.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
    )
    try:
        mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd_m, depth=8)
        mesh = mesh.crop(pcd_m.get_axis_aligned_bounding_box())
        leaf_area_m2 = float(mesh.get_surface_area())
    except Exception as e:
        print(f"  叶面积估算失败: {e}")
        leaf_area_m2 = -1.0

    return {
        "株高_m":    round(height_m,     4),
        "冠幅_m":    round(canopy_m,     4),
        "体积_m3":   round(volume_m3,    6),
        "叶面积_m2": round(leaf_area_m2, 4),
    }


## 主流程：逐PLY处理 → CSV / Excel 输出

In [ ]:
import pandas as pd
import traceback

records    = []
ply_files  = list(DATASET_DIR.glob("*.ply"))
print(f"找到 {len(ply_files)} 个PLY文件")

for ply_path in tqdm(ply_files):
    sample_id = ply_path.stem
    row = {"样本ID": sample_id}
    try:
        pcd = o3d.io.read_point_cloud(str(ply_path))
        print(f"\n[{sample_id}] 点数={len(pcd.points)}")

        # 1. 尺度校准
        scale = calibrate_scale_aruco(
            str(ply_path),
            marker_size_m=ARUCO_MARKER_SIZE_M,
            dict_name=ARUCO_DICT,
        )
        row["缩放因子"] = round(scale, 5)

        # 2. 地面识别
        plane_model, _ground, above = detect_ground(pcd, distance_threshold=0.02)

        # 3. 植物分割
        plant_pcd = segment_plant(above, eps=0.05, min_points=20)
        o3d.io.write_point_cloud(str(RESULT_DIR / f"{sample_id}_plant.ply"), plant_pcd)

        # 4. 指标计算
        traits = compute_traits(plant_pcd, plane_model, scale)
        row.update(traits)
        print(f"  {traits}")

    except Exception as e:
        print(f"  [错误] {e}")
        traceback.print_exc()
        row["错误"] = str(e)

    records.append(row)

df = pd.DataFrame(records)
df.to_csv(str(RESULT_DIR / "plant_traits.csv"),   index=False, encoding="utf-8-sig")
df.to_excel(str(RESULT_DIR / "plant_traits.xlsx"), index=False)
print(f"\n完成！结果: {RESULT_DIR}")
df
